In [2]:
import sys, os

# ── Fix 1: make `src` importable ──────────────────────────────────────────────
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# ── Fix 2: move CWD to project root so relative paths in config.yaml work ────
os.chdir(project_root)

print("CWD:", os.getcwd())       # should be: .../Loan-Risk-Intelligence
print("sys.path[0]:", sys.path[0])

from src.llm.generate_dataset import *


CWD: c:\college\PROJECTS\ML\Loan-Risk-Intelligence
sys.path[0]: c:\college\PROJECTS\ML\Loan-Risk-Intelligence
Saved 1000 records to C:\college\PROJECTS\ML\Loan-Risk-Intelligence\data\outputs\shap_dataset_raw.json
High: 400 | Medium: 200 | Low: 400
CSV preview saved.


In [2]:
# Quick test cell in notebook
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

test = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Say: API key working."}],
    max_tokens=10
)
print(test.choices[0].message.content)


API key working.


In [3]:
from src.llm.generate_explanations import generate_explanations
from pathlib import Path
from src.utils.config import load_config, get_project_root

config     = load_config()
output_dir = Path(get_project_root()) / config["paths"]["outputs"]

# Test 50 before full regeneration
generate_explanations(
    input_path  = output_dir / "shap_dataset_raw.json",
    output_path = output_dir / "shap_dataset_explained_v2.json",
    model       = "gpt-4o",
)



Progress: 10/1000 | Last: The loan exhibits a high default probability of 74.7%, placing it in the high-ri...
Progress: 20/1000 | Last: The loan has a default probability of 67.3% and falls into the High-risk tier. T...
Progress: 30/1000 | Last: The loan has a default probability of 75.2%, placing it in the high-risk tier. T...
Progress: 40/1000 | Last: Default probability is 70.2%, placing the loan in the high-risk tier. The primar...
Progress: 50/1000 | Last: The loan has an 82.8% default probability, placing it in the High risk tier. The...
Progress: 60/1000 | Last: Default probability is 69.8%, placing the loan in a high-risk tier. The primary ...
Progress: 70/1000 | Last: Default probability stands at 71.7%, placing the loan in the High risk tier. The...
Progress: 80/1000 | Last: Default probability is 75.2%, placing the loan in the high-risk tier. The primar...
Progress: 90/1000 | Last: Default probability is 73.1%, placing the loan in the High-risk tier. The primar...
Progress: 

[{'loan_id': 278193,
  'default_prob': 0.6752,
  'risk_tier': 'High',
  'actual_default': 0,
  'shap_drivers': [{'feature': 'FEDFUNDS_resid',
    'shap_value': 0.4008,
    'direction': 'increases_risk',
    'raw_value': 0.1869},
   {'feature': 'CPIUS_resid',
    'shap_value': 0.3366,
    'direction': 'increases_risk',
    'raw_value': 2.5853},
   {'feature': 'muni_points_resid',
    'shap_value': 0.2842,
    'direction': 'increases_risk',
    'raw_value': -0.2743},
   {'feature': 'term_',
    'shap_value': 0.2297,
    'direction': 'increases_risk',
    'raw_value': 1.0},
   {'feature': 'acc_open_past_24mths',
    'shap_value': -0.1982,
    'direction': 'reduces_risk',
    'raw_value': 0.0}],
  'macro_context': {'FEDFUNDS_resid': 0.1869,
   'CPIUS_resid': 2.5853,
   'inf_resid': -0.5439,
   'riskprem_resid': -0.0101,
   'FED_lag6_resid': 0.1675,
   'muni_6m_resid': -0.2816},
  'issue_year': 2014,
  'explanation': 'The default probability is 67.5%, placing this loan in the High risk tier

In [3]:
import json
from pathlib import Path
from src.utils.config import load_config, get_project_root

config     = load_config()
output_dir = Path(get_project_root()) / config["paths"]["outputs"]

with open(output_dir / "shap_dataset_explained_v2.json") as f:
    data = json.load(f)

empty = sum(1 for r in data if not r.get("explanation", "").strip())
print(f"Total records: {len(data)}")
print(f"Empty explanations: {empty}")
print(f"\nSample:\n{data[0]['explanation']}")


Total records: 1000
Empty explanations: 0

Sample:
The default probability is 67.5%, placing this loan in the High risk tier. The primary SHAP driver is FEDFUNDS_resid with a value of 0.1869, indicating a tightening-era risk where above-trend Fed rates compress borrower cash flows, increasing debt-service burden. CPIUS_resid also contributes significantly with a value of 2.5853, reflecting an inflationary squeeze that erodes real income, heightening financial stress.


In [4]:
import json
import random
from pathlib import Path
from src.utils.config import load_config, get_project_root

config     = load_config()
output_dir = Path(get_project_root()) / config["paths"]["outputs"]

with open(output_dir / "shap_dataset_explained_v2.json") as f:
    records = json.load(f)

records = [r for r in records if r.get("explanation", "").strip()]

def format_alpaca(record):
    drivers = record["shap_drivers"][:3]
    macro   = record["macro_context"]
    driver_str = "; ".join([
        f"{d['feature']} (SHAP {d['shap_value']:+.4f}, {'↑risk' if d['direction']=='increases_risk' else '↓risk'})"
        for d in drivers
    ])
    macro_str = ", ".join([f"{k}: {v:+.4f}" for k, v in list(macro.items())[:3]])
    return {
        "instruction": "You are a senior credit risk analyst. Generate a concise 2-3 sentence professional risk explanation based on the model output provided.",
        "input": f"Default Probability: {record['default_prob']:.1%} | Risk Tier: {record['risk_tier']} | Issue Year: {record['issue_year']}\nTop SHAP Drivers: {driver_str}\nMacro Context: {macro_str}",
        "output": record["explanation"]
    }

formatted = [format_alpaca(r) for r in records]

random.seed(42)
random.shuffle(formatted)
train = formatted[:800]
val   = formatted[800:900]
test  = formatted[900:]

def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl(train, output_dir / "train.jsonl")
save_jsonl(val,   output_dir / "val.jsonl")
save_jsonl(test,  output_dir / "test.jsonl")

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"\nSample input:\n{train[0]['input']}")
print(f"\nSample output:\n{train[0]['output']}")


Train: 800 | Val: 100 | Test: 100

Sample input:
Default Probability: 0.2% | Risk Tier: Low | Issue Year: 2018
Top SHAP Drivers: FEDFUNDS_resid (SHAP -2.3688, ↓risk); inf_6m_resid (SHAP -0.4534, ↓risk); inf_resid (SHAP -0.2709, ↓risk)
Macro Context: FEDFUNDS_resid: -0.0007, CPIUS_resid: +0.0000, inf_resid: +0.1429

Sample output:
The loan has a default probability of 0.2%, categorizing it in the Low risk tier. The primary SHAP driver is FEDFUNDS_resid with a value of -0.0007, indicating an accommodation-era risk where below-trend Fed rates reduce rollover risk, thereby lowering delinquency spikes upon policy normalization. The inf_6m_resid, with a SHAP value of -0.4534 and a value of 0.2428, further mitigates risk by indicating recent low inflation pressures.


In [5]:
import torch
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")


NVIDIA GeForce RTX 3060 Laptop GPU
6.4 GB VRAM


In [6]:
!pip install unsloth trl datasets -q


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\college\\PROJECTS\\ML\\Loan-Risk-Intelligence\\loanvenv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
